# 🎙️ Speech Dynamics Studio — Gradio Demo
 
**Model:** `MotionEmoBackboneV10` — 512-dim Transformer + 3 `StateHead` (Activation / Tension / Stability)  
**Demo này:** Tầng ứng dụng — chạy bên trên phần nghiên cứu đã đóng băng, không thay đổi ý nghĩa khoa học.

---
### Hai nhánh tách biệt tuyệt đối
| Nhánh | Input | Đo gì |
|---|---|---|
| **MODEL (A/T/S)** | Waveform chuẩn hóa −23 LUFS | *Cách nói*: sôi nổi / căng / đều — **KHÔNG bao giờ nói to/nhỏ** |
| **RAW (độ lớn thật)** | Waveform gốc | Độ lớn giọng nói thực (RMS − noise floor) — **nguồn DUY NHẤT** kết luận to/nhỏ |

---

In [ ]:
# ============================================================
# CELL 1 — Cài đặt thư viện
# ============================================================
!pip install -q gradio torch torchaudio numpy pandas scipy soundfile librosa pyloudnorm matplotlib tqdm
print('✅ Thư viện đã cài xong')


In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã mount')

In [ ]:
# ============================================================
# CELL 3 — Imports & cấu hình
# ============================================================
import os, json, math, random, warnings, hashlib, io, tempfile
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import soundfile as sf
import librosa
import pyloudnorm as pyln
from tqdm.auto import tqdm

import matplotlib
matplotlib.use('Agg')   # non-interactive backend cho Gradio
import matplotlib.pyplot as plt
import matplotlib as mpl
import gradio as gr

warnings.filterwarnings('ignore')

# ── Đường dẫn checkpoint (chỉnh ở đây) ──────────────────────
DRIVE_ROOT       = Path('/content/drive/MyDrive')
PHASE1_CKPT_PATH = DRIVE_ROOT / 'motionemo_v10_phase1/checkpoints/best_phase1_v10.pt'
PHASE2_CKPT_PATH = DRIVE_ROOT / 'motionemo_v10_phase2/checkpoints/best_phase2_state.pt'
# ─────────────────────────────────────────────────────────────

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP_OK    = DEVICE == 'cuda'

STATE_NAMES        = ['activation', 'tension', 'stability']
STATE_SCORE_LEVELS = np.array([0.0, 0.5, 1.0], dtype=np.float32)

AUDIO_CFG = dict(
    sample_rate=16000, n_mels=80, n_fft=512, win_length=400,
    hop_length=160, f_min=50.0, f_max=8000.0,
    f0_min=50.0, f0_max=500.0, top_db=80.0,
    target_lufs=-23.0, peak_guard=0.98,
)
WIN_CFG = dict(window_sec=2.5, hop_sec=0.5, min_window_sec=1.2)

DEMO_CFG = dict(
    smooth_window=5,
    high_tension_threshold=0.65,
    low_stability_threshold=0.35,
    stable_tension_threshold=0.25,
    stable_stability_threshold=0.70,
    rapid_transition_quantile=0.85,
    vad_top_db=35,
)

COLORS = dict(
    activation='#2F80ED', tension='#E76F51', stability='#2A9D8F',
    transition='#7B2CBF', grid='#DEE2E6', text='#212529',
)

print(f'Device: {DEVICE}  |  AMP: {AMP_OK}')
print(f'Phase1 ckpt exists: {PHASE1_CKPT_PATH.exists()}')
print(f'Phase2 ckpt exists: {PHASE2_CKPT_PATH.exists()}')

In [ ]:
# ============================================================
# CELL 4 — Kiến trúc model (giống hệt nghiên cứu, KHÔNG sửa)
# ============================================================
def make_padding_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    idx = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)


class MotionEmoBackboneV10(nn.Module):
    def __init__(self, n_mels=80, embed_dim=512, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.prosody_encoder = nn.Sequential(
            nn.Conv1d(3, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.mel_encoder = nn.Sequential(
            nn.Conv1d(n_mels, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.fusion_proj = nn.Sequential(nn.Linear(512, embed_dim), nn.LayerNorm(embed_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.attn_pool = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4), nn.Tanh(), nn.Linear(embed_dim // 4, 1)
        )

    def forward(self, mel, f0, energy, srate, lengths=None):
        B, _, T = mel.shape
        prosody_in = torch.stack([f0, energy, srate], dim=1)
        p_feat = self.prosody_encoder(prosody_in)
        m_feat = self.mel_encoder(mel)
        x = self.fusion_proj(torch.cat([p_feat, m_feat], dim=1).transpose(1, 2))
        pad_mask = make_padding_mask(lengths, T) if lengths is not None else None
        H_frame = self.transformer(x, src_key_padding_mask=pad_mask)
        attn_logits = self.attn_pool(H_frame).squeeze(-1)
        if pad_mask is not None:
            attn_logits = attn_logits.masked_fill(pad_mask, -1e4)
        attn = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        return H_frame, (attn * H_frame).sum(dim=1)


class StateHead(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, H_utt):
        return self.net(H_utt)


print('✅ Kiến trúc model đã định nghĩa')

In [ ]:
# ============================================================
# CELL 5 — Load checkpoint
# ============================================================
def _load_ckpt(path: Path):
    return torch.load(path, map_location=DEVICE, weights_only=False)


def _build_backbone(ckpt: dict):
    cfg = ckpt.get('cfg', {})
    mc  = cfg.get('model', {}) if isinstance(cfg, dict) else {}
    ed  = int(mc.get('embedding_dim', 512))
    bb  = MotionEmoBackboneV10(
        n_mels=AUDIO_CFG['n_mels'],
        embed_dim=ed,
        num_heads=int(mc.get('num_heads', 8)),
        num_layers=int(mc.get('num_layers', 6)),
        dropout=float(mc.get('dropout', 0.1)),
    ).to(DEVICE)
    bb.load_state_dict(ckpt['backbone'], strict=True)
    bb.eval()
    sh = int(mc.get('state_hidden', 256))
    nc = int(mc.get('num_state_classes', 3))
    return bb, ed, sh, nc


# Load Phase 2 (A/T/S heads)
p2_ckpt = _load_ckpt(PHASE2_CKPT_PATH)
phase2_bb, embed_dim, state_hidden, n_classes = _build_backbone(p2_ckpt)

act_head = StateHead(embed_dim, state_hidden, n_classes).to(DEVICE)
ten_head = StateHead(embed_dim, state_hidden, n_classes).to(DEVICE)
sta_head = StateHead(embed_dim, state_hidden, n_classes).to(DEVICE)
act_head.load_state_dict(p2_ckpt['activation_head']); act_head.eval()
ten_head.load_state_dict(p2_ckpt['tension_head']);    ten_head.eval()
sta_head.load_state_dict(p2_ckpt['stability_head']); sta_head.eval()

# Load Phase 1 (embedding cho boundary detection)
p1_ckpt    = _load_ckpt(PHASE1_CKPT_PATH)
phase1_bb, *_ = _build_backbone(p1_ckpt)

print(f'✅ Phase 2 loaded  |  epoch={p2_ckpt.get("epoch")}  |  best_state_acc={p2_ckpt.get("best_state_acc")}')
print(f'✅ Phase 1 loaded  |  embed_dim={embed_dim}')

In [ ]:
# ============================================================
# CELL 6 — Engine: tiện ích & audio processing
# ============================================================
def autocast_ctx():
    return torch.amp.autocast('cuda', enabled=True) if AMP_OK else nullcontext()

def rolling_smooth(arr, w=5):
    return pd.Series(arr).rolling(w, center=True, min_periods=1).mean().values

def safe_zscore(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x)
    mu, sd = x.mean(), x.std()
    return (x - mu) / (max(sd, eps))

def align_to_T(x, T):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) >= T: return x[:T]
    return np.concatenate([x, np.full(T - len(x), x[-1] if len(x) else 0.0)])


# ── Nhánh RAW: chuẩn hóa LUFS (sao chép chính xác từ pipeline nghiên cứu) ──
def normalize_lufs(wav: np.ndarray, sr: int, target=-23.0, peak_guard=0.98) -> np.ndarray:
    """Chuẩn hóa LUFS cho nhánh MODEL — dùng wav_model."""
    if len(wav) < int(0.4 * sr):
        return wav.astype(np.float32)
    peak = np.max(np.abs(wav)) + 1e-12
    rms  = np.sqrt(np.mean(wav**2) + 1e-12)
    if peak < 1e-5 or rms < 1e-6:
        return wav.astype(np.float32)
    try:
        meter = pyln.Meter(sr)
        lufs  = meter.integrated_loudness(wav.astype(np.float64))
        if not np.isfinite(lufs) or lufs < -60:
            return wav.astype(np.float32)
    except Exception:
        return wav.astype(np.float32)
    gain_db = np.clip(target - lufs, -24.0, 24.0)
    y = wav * (10 ** (gain_db / 20.0))
    pk = np.max(np.abs(y))
    if pk > peak_guard:
        y = y * (peak_guard / (pk + 1e-8))
    return y.astype(np.float32)


def load_audio(path_or_bytes, sr=16000):
    """Trả về (wav_raw, wav_model, sr)."""
    if isinstance(path_or_bytes, (str, Path)):
        wav, native_sr = sf.read(str(path_or_bytes), always_2d=False)
    else:
        buf = io.BytesIO(path_or_bytes)
        wav, native_sr = sf.read(buf, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = np.nan_to_num(wav.astype(np.float32))
    if native_sr != sr:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=sr).astype(np.float32)
    wav_raw   = wav.copy()
    wav_model = normalize_lufs(wav, sr)
    return wav_raw, wav_model, sr


def extract_features(wav: np.ndarray, sr: int) -> dict:
    """Trích đặc trưng cho một cửa sổ (dùng cho nhánh MODEL)."""
    sr_cfg = AUDIO_CFG
    mel_pow = librosa.feature.melspectrogram(
        y=wav, sr=sr, n_fft=sr_cfg['n_fft'], hop_length=sr_cfg['hop_length'],
        win_length=sr_cfg['win_length'], n_mels=sr_cfg['n_mels'],
        fmin=sr_cfg['f_min'], fmax=sr_cfg['f_max'], power=2.0,
    )
    mel = safe_zscore(librosa.power_to_db(mel_pow, ref=np.max, top_db=sr_cfg['top_db']))
    T   = mel.shape[1]

    rms = librosa.feature.rms(y=wav, frame_length=sr_cfg['win_length'],
                               hop_length=sr_cfg['hop_length'])[0].astype(np.float32)
    zcr = librosa.feature.zero_crossing_rate(y=wav, frame_length=sr_cfg['win_length'],
                                              hop_length=sr_cfg['hop_length'])[0].astype(np.float32)
    try:
        f0 = librosa.yin(wav, fmin=sr_cfg['f0_min'], fmax=sr_cfg['f0_max'], sr=sr,
                         frame_length=sr_cfg['win_length'], hop_length=sr_cfg['hop_length']).astype(np.float32)
    except Exception:
        f0 = np.zeros(T, dtype=np.float32)

    rms = align_to_T(rms, T); zcr = align_to_T(zcr, T); f0 = align_to_T(np.nan_to_num(f0), T)

    rms_p95 = np.percentile(rms, 95) + 1e-8
    voiced  = (rms / rms_p95 > 0.03) & (f0 > 0)
    f0_norm = np.zeros_like(f0)
    if voiced.sum() > 1:
        f0_norm[voiced] = safe_zscore(f0[voiced])

    return dict(
        mel=torch.from_numpy(mel.astype(np.float32)),
        f0=torch.from_numpy(f0_norm.astype(np.float32)),
        energy=torch.from_numpy(safe_zscore(rms).astype(np.float32)),
        srate=torch.from_numpy(safe_zscore(zcr).astype(np.float32)),
        energy_raw=rms, num_frames=T,
    )


def make_windows(wav: np.ndarray, sr: int):
    """
    Tạo danh sách (t_start, t_end, chunk) theo lưới trượt (window, hop).

    LƯU Ý QUAN TRỌNG: vòng lặp trượt chuẩn (s += hop cho đến khi
    s + win >= len(wav)) khiến cửa sổ cuối cùng dừng lại sớm — center
    của điểm dữ liệu cuối chỉ đạt khoảng len(wav) - win/2, bỏ sót gần
    nửa window (~1.25s với window 2.5s) ở đuôi file. Vì mọi biểu đồ
    (Voice Level RAW, Activation/Tension/Stability) đều vẽ theo `center`
    của các cửa sổ này, hệ quả là các đường đó kết thúc sớm hơn hẳn so
    với waveform (vốn vẽ trực tiếp theo toàn bộ mẫu âm thanh).

    Khắc phục: sau khi tạo lưới trượt như cũ, LUÔN thêm một cửa sổ
    "neo cuối" (anchor cuối) có t_end = len(wav) nếu cửa sổ cuối cùng
    hiện tại chưa chạm tới cuối file — đảm bảo center cuối cùng tiệm
    cận hết chiều dài audio, khớp với waveform.
    """
    win = int(WIN_CFG['window_sec'] * sr)
    hop = int(WIN_CFG['hop_sec'] * sr)
    min_len = int(WIN_CFG['min_window_sec'] * sr)
    n = len(wav)
    out = []
    if n < min_len:
        return out
    if n <= win:
        out.append((0, n, wav.copy()))
        return out

    s = 0
    while s < n:
        e = s + win
        chunk = wav[s:min(e, n)]
        if len(chunk) >= min_len:
            out.append((s, min(e, n), chunk.copy()))
        if e >= n:
            break
        s += hop

    # Neo cửa sổ cuối tới đúng cuối file nếu vòng lặp trên dừng sớm
    last_end = out[-1][1] if out else 0
    if last_end < n:
        anchor_start = max(0, n - win)
        if anchor_start >= min_len or n - anchor_start >= min_len:
            out.append((anchor_start, n, wav[anchor_start:n].copy()))

    return out


print('✅ Engine utils định nghĩa xong')

In [ ]:
# ============================================================
# CELL 7 — Engine: VAD & Raw branch & Model branch
# ============================================================

# ── VAD ─────────────────────────────────────────────────────
def detect_speech_segments(wav_model: np.ndarray, sr: int) -> List[Tuple[float, float]]:
    """VAD thích nghi, trả về list (t_start, t_end) giây."""
    sr_cfg = AUDIO_CFG
    intervals = librosa.effects.split(
        wav_model, top_db=DEMO_CFG['vad_top_db'],
        frame_length=sr_cfg['win_length'], hop_length=sr_cfg['hop_length'],
    )
    segs = []
    for s, e in intervals:
        # thêm buffer 100ms
        s2 = max(0, s - int(0.1 * sr))
        e2 = min(len(wav_model), e + int(0.1 * sr))
        segs.append((s2 / sr, e2 / sr))
    # Merge segments gần nhau (< 0.3s)
    merged = []
    for seg in segs:
        if merged and seg[0] - merged[-1][1] < 0.3:
            merged[-1] = (merged[-1][0], seg[1])
        else:
            merged.append(list(seg))
    return [(s, e) for s, e in merged]


def is_speech(t_start: float, speech_segs: List[Tuple[float, float]]) -> bool:
    """Tiêu chí chặt: t_start nằm trong một đoạn giọng nói."""
    for s, e in speech_segs:
        if s <= t_start < e:
            return True
    return False


def has_content(chunk: np.ndarray, sr: int) -> bool:
    """Tiêu chí lỏng: có đủ năng lượng để chạy model."""
    rms = np.sqrt(np.mean(chunk**2) + 1e-12)
    return rms > 5e-4


# ── Nhánh RAW ────────────────────────────────────────────────
def compute_raw_branch(wav_raw: np.ndarray, sr: int, windows: list) -> dict:
    """
    Chạy trên wav_raw (chưa chuẩn hóa).
    Tính voice_level_db = RMS_dB − noise_floor.
    NGUỒN DUY NHẤT được phép kết luận to/nhỏ.
    """
    sr_cfg = AUDIO_CFG
    hop = sr_cfg['hop_length']

    # Noise floor = percentile-10 RMS toàn file (trừ sàn nhiễu)
    rms_full = librosa.feature.rms(y=wav_raw, frame_length=sr_cfg['win_length'],
                                    hop_length=hop)[0]
    rms_db_full = 20 * np.log10(rms_full + 1e-12)
    noise_floor = float(np.percentile(rms_db_full, 10))

    raw_rows = []
    for t_start, t_end, _ in windows:
        chunk_raw = wav_raw[t_start:t_end]
        if len(chunk_raw) == 0:
            raw_rows.append({'raw_rms_db': noise_floor, 'voice_level_db': 0.0})
            continue
        rms_win = librosa.feature.rms(y=chunk_raw, frame_length=sr_cfg['win_length'],
                                       hop_length=hop)[0]
        rms_db  = float(np.mean(20 * np.log10(rms_win + 1e-12)))
        voice_level = max(0.0, rms_db - noise_floor)
        raw_rows.append({'raw_rms_db': rms_db, 'voice_level_db': voice_level})

    raw_df = pd.DataFrame(raw_rows)

    # Ngưỡng to/nhỏ tương đối nội bộ file
    vl = raw_df['voice_level_db'].values
    thresh_loud  = float(np.percentile(vl[vl > 0], 75)) if (vl > 0).sum() > 0 else 1.0
    thresh_quiet = float(np.percentile(vl[vl > 0], 30)) if (vl > 0).sum() > 0 else 0.0

    return dict(raw_df=raw_df, noise_floor=noise_floor,
                thresh_loud=thresh_loud, thresh_quiet=thresh_quiet)


# ── Nhánh MODEL ──────────────────────────────────────────────
@torch.no_grad()
def run_model_branch(windows: list, sr: int, speech_segs: list, batch_size=16):
    """
    Chạy trên windows từ wav_model (đã chuẩn hóa LUFS).
    Trả về DataFrame A/T/S + Phase-1 embeddings.
    """
    all_feats, all_times, all_flags = [], [], []

    for t_start, t_end, chunk in tqdm(windows, desc='Extract features', leave=False):
        t_s = t_start / sr
        all_times.append(dict(t_start=t_s, t_end=t_end/sr, center=(t_s + t_end/sr)/2))
        all_flags.append(dict(
            has_content=has_content(chunk, sr),
            is_speech=is_speech(t_s, speech_segs),
        ))
        all_feats.append(extract_features(chunk, sr))

    rows, p1_embs, p2_embs = [], [], []

    for i in range(0, len(all_feats), batch_size):
        batch = all_feats[i:i+batch_size]
        flags = all_flags[i:i+batch_size]
        active = [j for j, f in enumerate(flags) if f['has_content']]

        # Khởi tạo zeros
        for j in range(len(batch)):
            rows.append(dict(activation_score=0.5, tension_score=0.5, stability_score=0.5,
                             has_content=flags[j]['has_content'], is_speech=flags[j]['is_speech']))
            p1_embs.append(np.zeros(embed_dim, np.float32))
            p2_embs.append(np.zeros(embed_dim, np.float32))

        if not active:
            continue

        sub = [batch[j] for j in active]
        max_T = max(x['num_frames'] for x in sub)
        mels, f0s, energies, srates, lengths = [], [], [], [], []
        for x in sub:
            L = x['num_frames']; pad = max_T - L
            mels.append(F.pad(x['mel'], (0, pad)).unsqueeze(0))
            f0s.append(F.pad(x['f0'], (0, pad)).unsqueeze(0))
            energies.append(F.pad(x['energy'], (0, pad)).unsqueeze(0))
            srates.append(F.pad(x['srate'], (0, pad)).unsqueeze(0))
            lengths.append(L)

        mel_t = torch.cat(mels).to(DEVICE)
        f0_t  = torch.cat(f0s).to(DEVICE)
        en_t  = torch.cat(energies).to(DEVICE)
        sr_t  = torch.cat(srates).to(DEVICE)
        len_t = torch.tensor(lengths, dtype=torch.long, device=DEVICE)

        with autocast_ctx():
            _, h1 = phase1_bb(mel_t, f0_t, en_t, sr_t, lengths=len_t)
            _, h2 = phase2_bb(mel_t, f0_t, en_t, sr_t, lengths=len_t)
            ap = F.softmax(act_head(h2), dim=-1)
            tp = F.softmax(ten_head(h2), dim=-1)
            sp = F.softmax(sta_head(h2), dim=-1)

        h1_np = h1.float().cpu().numpy()
        h2_np = h2.float().cpu().numpy()
        ap_np = ap.float().cpu().numpy()
        tp_np = tp.float().cpu().numpy()
        sp_np = sp.float().cpu().numpy()

        for k, j in enumerate(active):
            global_idx = i + j
            rows[global_idx].update(dict(
                activation_score=float((ap_np[k] * STATE_SCORE_LEVELS).sum()),
                tension_score=float((tp_np[k] * STATE_SCORE_LEVELS).sum()),
                stability_score=float((sp_np[k] * STATE_SCORE_LEVELS).sum()),
            ))
            p1_embs[global_idx] = h1_np[k]
            p2_embs[global_idx] = h2_np[k]

    df = pd.DataFrame(all_times)
    df = pd.concat([df, pd.DataFrame(rows)], axis=1)

    # Smooth chỉ trên cửa sổ IS_SPEECH
    for s in STATE_NAMES:
        vals = df[f'{s}_score'].where(df['is_speech'], np.nan)
        df[f'{s}_smooth'] = vals.ffill().bfill()
        df[f'{s}_smooth'] = rolling_smooth(df[f'{s}_smooth'].values, DEMO_CFG['smooth_window'])

    # State velocity
    for s in STATE_NAMES:
        df[f'd_{s}'] = df[f'{s}_smooth'].diff().fillna(0.0)
    df['state_velocity'] = np.sqrt(df['d_activation']**2 + df['d_tension']**2 + df['d_stability']**2)

    return df, np.array(p1_embs), np.array(p2_embs)


print('✅ VAD + Raw branch + Model branch định nghĩa xong')

In [ ]:
# ============================================================
# CELL 8 — Segment builder & Boundary detection (Tầng A)
# ============================================================
from scipy.spatial.distance import cosine as cosine_dist

# (5) Buffer chống nhảy nhãn: đoạn speech ngắn hơn ngưỡng này được đánh dấu
# để mô tả tự nhiên (Cell 9) ưu tiên xử lý như "chớm qua rồi quay lại"
# thay vì khẳng định cứng High/Low.
BUFFER_MIN_SEC_FOR_SEGMENTS = 0.5


def detect_acoustic_boundaries(df: pd.DataFrame, p1_embs: np.ndarray,
                                 percentile: float = 85.0) -> List[float]:
    """
    Tầng A: cosine distance giữa Phase-1 embeddings của các cửa sổ
    giọng nói liền kề thật. Ranh giới = > percentile-85 thích nghi.
    """
    speech_idx = df.index[df['is_speech']].tolist()
    dists = []
    pairs = []
    for a, b in zip(speech_idx[:-1], speech_idx[1:]):
        # Kiểm tra liền kề thật: không bắc cầu qua khoảng lặng
        if df.loc[a, 't_end'] >= df.loc[b, 't_start'] - 0.1:
            d = cosine_dist(
                p1_embs[a] + 1e-12, p1_embs[b] + 1e-12
            ) if np.any(p1_embs[a]) and np.any(p1_embs[b]) else 0.0
            dists.append(d)
            pairs.append((a, b))

    if not dists:
        return []

    threshold = np.percentile(dists, percentile)
    boundaries = []
    for (a, b), d in zip(pairs, dists):
        if d >= threshold:
            boundaries.append(float(df.loc[a, 't_end']))
    return boundaries


def build_segments(df: pd.DataFrame, raw_info: dict,
                   speech_segs: list, boundaries: list) -> List[dict]:
    """
    Chia timeline thành đoạn theo khoảng nghỉ VAD + ranh giới Tầng A.
    Mỗi đoạn chứa trung bình CẢ hai nhánh.
    """
    raw_df = raw_info['raw_df']

    # Thu thập tất cả cut-points
    cuts = set()
    for s, e in speech_segs:
        cuts.add(s); cuts.add(e)
    for b in boundaries:
        cuts.add(b)
    cuts = sorted(cuts)

    t_min = float(df['t_start'].min())
    t_max = float(df['t_end'].max())
    edges = [t_min] + [c for c in cuts if t_min < c < t_max] + [t_max]

    segments = []
    for i in range(len(edges) - 1):
        seg_start, seg_end = edges[i], edges[i+1]
        mask = (df['center'] >= seg_start) & (df['center'] < seg_end)
        if mask.sum() == 0:
            continue
        sub = df[mask]
        sub_raw = raw_df[mask]
        is_sp   = bool(sub['is_speech'].any())
        seg = dict(
            t_start=seg_start, t_end=seg_end,
            duration=seg_end - seg_start,
            is_speech=is_sp,
            is_boundary=any(abs(seg_start - b) < 0.05 for b in boundaries),
            n_windows=int(mask.sum()),
        )
        for st in STATE_NAMES:
            seg[f'{st}_mean'] = float(sub[f'{st}_smooth'].mean())
        seg['voice_level_db_mean'] = float(sub_raw['voice_level_db'].mean())
        seg['state_velocity_max']  = float(sub['state_velocity'].max())

        # (5) Buffer: đoạn speech quá ngắn → coi như chưa đủ để xác nhận High/Low
        seg['is_buffer_short'] = bool(is_sp and seg['duration'] < BUFFER_MIN_SEC_FOR_SEGMENTS)
        segments.append(seg)

    return segments


print('✅ Segment builder & Boundary detection định nghĩa xong')

In [ ]:
# ============================================================
# CELL 9 — Ánh xạ động học giọng nói (Vocal Dynamics Mapping)
# ============================================================
# Hai nhánh TÁCH BẠCH:
#   • MANNER_PATTERNS — khóa theo (activation, tension, stability), mô tả
#     CÁCH NÓI (sôi nổi / căng / đều). Phủ ĐỦ 27 tổ hợp, KHÔNG nhắc to/nhỏ.
#   • RAW_QUALIFIER  — tiền tố to/nhỏ lấy DUY NHẤT từ nhánh RAW.
# Mô tả cuối = [tiền tố RAW] + [mẫu hình cách nói]  → không trộn ý nghĩa.
#
# Xử lý vùng Mid (giữ nguyên ý tưởng): Mid là "Trạng thái Cân bằng" có
# mô tả riêng (tổ hợp mid/mid/mid), đo Dynamics Salience, gợi ý xu hướng
# khi điểm gần ngưỡng, nhận diện quỹ đạo so với đoạn trước, và buffer 0.5s.

MID_LO, MID_HI      = 0.35, 0.65
NEAR_THRESHOLD_BAND = 0.10
BUFFER_MIN_SEC      = 0.5


def _level3(v: float, lo=MID_LO, hi=MID_HI) -> str:
    if v >= hi: return 'high'
    if v <= lo: return 'low'
    return 'mid'

def _level_raw(vl: float, thresh_loud: float, thresh_quiet: float) -> str:
    if vl >= thresh_loud:  return 'high'
    if vl <= thresh_quiet: return 'low'
    return 'mid'

def _near_threshold_hint(v, lo=MID_LO, hi=MID_HI, band=NEAR_THRESHOLD_BAND):
    """Điểm Mid nhưng sát biên → trả 'rising'/'falling' để dùng trạng từ giảm nhẹ."""
    if hi - band <= v < hi: return 'rising'
    if lo < v <= lo + band: return 'falling'
    return None

def compute_dynamics_salience(a_lvl, t_lvl, s_lvl) -> str:
    """Độ 'nổi bật' động học: low = cả 3 trục Mid (đệm), high = cả 3 lệch Mid."""
    n_mid = [a_lvl, t_lvl, s_lvl].count('mid')
    if n_mid == 3: return 'low'
    if n_mid == 0: return 'high'
    return 'medium'

def classify_trajectory(prev_seg, cur_seg):
    """Quỹ đạo trạng thái tổng hợp so với đoạn nói liền trước."""
    if prev_seg is None or not prev_seg.get('is_speech') or not cur_seg.get('is_speech'):
        return None
    pl = _level3((prev_seg['activation_mean'] + prev_seg['tension_mean'] + prev_seg['stability_mean']) / 3)
    cl = _level3((cur_seg['activation_mean']  + cur_seg['tension_mean']  + cur_seg['stability_mean'])  / 3)
    return {
        ('mid', 'mid'): 'Ổn định kéo dài', ('mid', 'high'): 'Gia tăng sắc thái',
        ('mid', 'low'): 'Lắng xuống dần', ('high', 'mid'): 'Hạ nhiệt về cân bằng',
        ('low', 'mid'): 'Phục hồi về cân bằng', ('low', 'high'): 'Bứt phá mạnh',
        ('high', 'low'): 'Sụt giảm đột ngột',
    }.get((pl, cl))


# ── Tiền tố độ lớn — lấy DUY NHẤT từ nhánh RAW ──────────────────────────
RAW_QUALIFIER = {
    'high': {'vi': 'Nói to, vang',     'en': 'loud, projected'},
    'low':  {'vi': 'Nói khẽ, nhỏ',     'en': 'quiet, soft'},
    'mid':  {'vi': 'Độ lớn vừa phải',  'en': 'moderate volume'},
}

# ── 27 mẫu hình CÁCH NÓI, khóa (activation, tension, stability) ─────────
# Mô tả gọn 1 câu, KHÔNG dùng từ to/nhỏ (đó là việc của nhánh RAW).
MANNER_PATTERNS = {
    ('high','high','high'): dict(vi_name='Cao trào & Giữ nhịp',
        vi='Sôi nổi và căng nhưng nhịp vẫn rất đều — như một cao trào có kiểm soát.',
        en='rising, intense yet tightly metered — a controlled climax'),
    ('high','high','mid'): dict(vi_name='Hăng & Dồn nhịp',
        vi='Nhiều năng lượng và độ căng, nhịp được giữ ở mức tương đối.',
        en='driving energy and tension, rhythm fairly held'),
    ('high','high','low'): dict(vi_name='Dồn dập & Xáo trộn',
        vi='Sôi nổi, căng, nhịp xô lệch liên tục — vận động cuồn cuộn, thiếu ổn định.',
        en='forceful and erratic, rhythm breaking up'),
    ('high','mid','high'): dict(vi_name='Sôi nổi & Vững nhịp',
        vi='Hào hứng, nhịp đều, độ căng vừa phải — cuốn hút mà chắc chắn.',
        en='lively and well-paced, moderate tension'),
    ('high','mid','mid'): dict(vi_name='Sôi nổi tự nhiên',
        vi='Nhiều năng lượng, độ căng và nhịp đều ở mức vừa — nói hứng khởi, thoải mái.',
        en='naturally lively, even tension and rhythm'),
    ('high','mid','low'): dict(vi_name='Hào hứng nhưng chông chênh',
        vi='Hào hứng nhưng nhịp thất thường — hứng lên mà chưa định hình được nhịp.',
        en='enthusiastic but rhythmically unsteady'),
    ('high','low','high'): dict(vi_name='Hào hứng & Thư thái',
        vi='Năng lượng cao, thả lỏng, nhịp đều — cuốn hút mà vẫn nhẹ nhàng.',
        en='energetic yet relaxed, steady rhythm'),
    ('high','low','mid'): dict(vi_name='Hào hứng nhẹ nhõm',
        vi='Nhiều năng lượng, độ căng thấp, nhịp khá ổn — nói hào hứng mà dễ chịu.',
        en='energetic and easy, fairly steady'),
    ('high','low','low'): dict(vi_name='Phóng khoáng & Tản nhịp',
        vi='Sôi nổi, thả lỏng nhưng nhịp lỏng lẻo — phóng khoáng, chưa gọn nhịp.',
        en='free-flowing and energetic but loosely paced'),

    ('mid','high','high'): dict(vi_name='Căng & Giữ nhịp',
        vi='Độ căng cao nhưng nhịp rất đều — kìm nén một cách có kỷ luật.',
        en='tense but rhythmically disciplined'),
    ('mid','high','mid'): dict(vi_name='Gấp gáp & Thúc giục',
        vi='Độ căng nổi lên, nhịp khá giữ — đẩy nhịp, có cảm giác hối thúc.',
        en='urgent, pushing the pace'),
    ('mid','high','low'): dict(vi_name='Căng & Bồn chồn',
        vi='Căng và nhịp chông chênh — bất an, gấp gáp nhưng không đều.',
        en='tense and restless, uneven pacing'),
    ('mid','mid','high'): dict(vi_name='Điềm đạm & Đều nhịp',
        vi='Các trục ở mức vừa, nhịp đều — nói khoan thai, chắc nhịp.',
        en='composed and evenly paced'),
    ('mid','mid','mid'): dict(vi_name='Cân bằng & Ổn định',
        vi='Trạng thái cân bằng, không có biến động âm học nổi bật — làm nền tham chiếu cho các đoạn khác.',
        en='balanced, neutral baseline — no salient shift'),
    ('mid','mid','low'): dict(vi_name='Bình thường & Lệch nhịp',
        vi='Các trục ở mức vừa nhưng nhịp hơi thất thường.',
        en='ordinary delivery with slightly uneven pacing'),
    ('mid','low','high'): dict(vi_name='Thư thái & Đều đặn',
        vi='Thả lỏng, nhịp đều, năng lượng vừa — nhẹ nhàng và mạch lạc.',
        en='relaxed and evenly paced, clear flow'),
    ('mid','low','mid'): dict(vi_name='Thư thái tự nhiên',
        vi='Thả lỏng, độ căng và nhịp ở mức vừa — thoải mái, đời thường.',
        en='relaxed and natural, moderate throughout'),
    ('mid','low','low'): dict(vi_name='Buông & Tản nhịp',
        vi='Thả lỏng nhưng nhịp lỏng — buông, hơi rời rạc.',
        en='loose and relaxed, unanchored pacing'),

    ('low','high','high'): dict(vi_name='Nén & Giữ chặt nhịp',
        vi='Ít nhấn nhá nhưng căng, nhịp đều — nén lại và giữ chặt nhịp.',
        en='restrained but tense, tightly held rhythm'),
    ('low','high','mid'): dict(vi_name='Trầm nhưng căng',
        vi='Ít nhấn nhá, độ căng nổi lên, nhịp khá giữ — kìm nén ngầm.',
        en='subdued yet tense, fairly steady'),
    ('low','high','low'): dict(vi_name='Trầm & Bất an',
        vi='Trầm, căng và nhịp chông chênh — căng thẳng âm ỉ, không đều.',
        en='subdued, tense and uneven'),
    ('low','mid','high'): dict(vi_name='Trầm & Đều',
        vi='Ít nhấn nhá, nhịp đều — bình thản, phẳng lặng.',
        en='subdued and even, flat and calm'),
    ('low','mid','mid'): dict(vi_name='Trầm lặng',
        vi='Ít năng lượng, độ căng và nhịp ở mức vừa — trầm và ít điểm nhấn.',
        en='subdued, low emphasis'),
    ('low','mid','low'): dict(vi_name='Trầm & Lỏng nhịp',
        vi='Ít nhấn nhá, nhịp thất thường — uể oải, hơi rời.',
        en='subdued with loose, uneven pacing'),
    ('low','low','high'): dict(vi_name='Phẳng lặng & Mượt',
        vi='Ít nhấn nhá, thả lỏng, nhịp rất đều — êm và trôi chảy.',
        en='flat and smooth, very steady'),
    ('low','low','mid'): dict(vi_name='Nhẹ nhàng & Bình thản',
        vi='Trầm, thư thái, nhịp khá ổn — thư giãn, đều đều.',
        en='gentle and calm, fairly steady'),
    ('low','low','low'): dict(vi_name='Uể oải & Rời rạc',
        vi='Trầm, thả lỏng, nhịp lỏng lẻo — đuối sức, thiếu điểm tựa nhịp.',
        en='weary and disjointed, loose pacing'),
}


def _levels_of(seg):
    return (_level3(seg['activation_mean']), _level3(seg['tension_mean']), _level3(seg['stability_mean']))

def _dominant_hint_vi(seg):
    """Một gợi ý xu hướng khi có trục Mid sát biên (nguyên tắc Continuous Scores)."""
    for v, rise, fall in [(seg['tension_mean'],   'độ căng đang nhích lên',  'độ căng đang dịu lại'),
                          (seg['activation_mean'],'càng lúc càng sôi nổi',   'đang trầm dần'),
                          (seg['stability_mean'], 'nhịp đang ổn định dần',   'nhịp đang chông chênh dần')]:
        h = _near_threshold_hint(v)
        if h == 'rising':  return f'({rise})'
        if h == 'falling': return f'({fall})'
    return None

def _tension_delta_en(prev_seg, seg):
    if prev_seg and prev_seg.get('is_speech'):
        dt = seg['tension_mean'] - prev_seg['tension_mean']
        if abs(dt) > 0.12:
            return f'(tension {"up" if dt > 0 else "down"} ~{int(abs(dt) * 100)}%)'
    return ''


def describe_segment_vi(seg, raw_info, prev_seg=None) -> str:
    if not seg['is_speech']:
        return f"[Khoảng lặng — {seg['duration']:.1f}s]"
    qual = RAW_QUALIFIER[_level_raw(seg['voice_level_db_mean'],
                                    raw_info['thresh_loud'], raw_info['thresh_quiet'])]['vi']
    if seg.get('is_buffer_short'):
        traj = classify_trajectory(prev_seg, seg)
        text = f'{qual}; đoạn thoáng qua, chưa đủ dài để xác nhận một xu hướng rõ rệt.'
        return text + (f' *(Quỹ đạo: {traj})*' if traj else '')

    a, t, s = _levels_of(seg)
    manner  = MANNER_PATTERNS[(a, t, s)]['vi']    # chỉ diễn giải, KHÔNG nêu tên mẫu hình
    text = f'{qual}; {manner[0].lower() + manner[1:]}'

    hint = _dominant_hint_vi(seg)
    if hint:
        text += f' {hint}'
    traj = classify_trajectory(prev_seg, seg)
    if traj:
        text += f' *(Quỹ đạo: {traj})*'
    return text


def describe_segment_en(seg, raw_info, prev_seg=None) -> str:
    if not seg['is_speech']:
        return f'[Pause {seg["duration"]:.1f}s]'
    qual = RAW_QUALIFIER[_level_raw(seg['voice_level_db_mean'],
                                    raw_info['thresh_loud'], raw_info['thresh_quiet'])]['en']
    if seg.get('is_buffer_short'):
        return f'{qual}; brief transitional moment — too short to confirm a pattern {_tension_delta_en(prev_seg, seg)}'.strip()
    a, t, s = _levels_of(seg)
    manner  = MANNER_PATTERNS[(a, t, s)]['en']
    return f'{qual}; {manner} {_tension_delta_en(prev_seg, seg)}'.strip()


def get_vocal_pattern_label(seg, raw_info) -> Optional[str]:
    """Tên mẫu hình cách nói (cho báo cáo/JSON)."""
    if not seg['is_speech']:
        return None
    a, t, s = _levels_of(seg)
    name = MANNER_PATTERNS[(a, t, s)]['vi_name']
    return name + ' (đệm)' if (a, t, s) == ('mid', 'mid', 'mid') else name

def get_dynamics_salience_label(seg) -> Optional[str]:
    if not seg['is_speech']:
        return None
    sal = compute_dynamics_salience(*_levels_of(seg))
    return {'low': 'Thấp (đệm)', 'medium': 'Trung bình', 'high': 'Cao (nổi bật)'}[sal]


print(f'✅ Ánh xạ: {len(MANNER_PATTERNS)} mẫu hình cách nói (phủ đủ 27 tổ hợp) + tiền tố độ lớn RAW')


In [ ]:
# ============================================================
# CELL 10 — Output generators (5 đầu ra)
# ============================================================

# ── Output 1: Timeline ───────────────────────────────────────
# Độ lớn (RAW) vẽ ở độ phân giải KHUNG trên toàn file → đúng điểm bắt đầu
# có tiếng và phủ trọn chiều dài như waveform. A/T/S vẽ theo TỪNG đoạn
# giọng nói (neo vào mốc VAD) → bắt đầu/kết thúc khớp với độ lớn, tự đứt ở
# khoảng lặng. Tất cả các trục dùng chung trục thời gian [0, duration].
def make_timeline_figure(df, wav_raw, sr, raw_info, speech_segs, boundaries) -> plt.Figure:
    duration = len(wav_raw) / sr
    fig, axes = plt.subplots(5, 1, figsize=(15, 10), sharex=True,
                             gridspec_kw={'height_ratios': [0.5, 0.8, 0.7, 0.7, 0.7]})
    fig.subplots_adjust(hspace=0.06)

    ax = axes[0]
    t_audio = np.arange(len(wav_raw)) / sr
    step = max(1, len(wav_raw) // 8000)
    ax.plot(t_audio[::step], wav_raw[::step], color='#ADB5BD', lw=0.6)
    ax.set_ylabel('Waveform', fontsize=9); ax.set_yticks([])
    ax.set_title('Speech Dynamics Studio — Timeline', fontsize=13, weight='bold')

    # Độ lớn RAW ở mức KHUNG (frame-level) trên toàn file
    hop, win = AUDIO_CFG['hop_length'], AUDIO_CFG['win_length']
    rms_f = librosa.feature.rms(y=wav_raw, frame_length=win, hop_length=hop)[0]
    vl_f  = np.maximum(0.0, 20 * np.log10(rms_f + 1e-12) - raw_info['noise_floor'])
    vl_f  = pd.Series(vl_f).rolling(9, center=True, min_periods=1).mean().values
    t_f   = librosa.frames_to_time(np.arange(len(vl_f)), sr=sr, hop_length=hop)
    ax1 = axes[1]
    ax1.fill_between(t_f, 0, vl_f, color='#495057', alpha=0.55)
    ax1.axhline(raw_info['thresh_loud'],  color='#C92A2A', lw=1.0, ls=':', label='Ngưỡng to')
    ax1.axhline(raw_info['thresh_quiet'], color='#2B8A3E', lw=1.0, ls=':', label='Ngưỡng nhỏ')
    ax1.set_ylabel('Độ lớn\ndB [RAW]', fontsize=8)
    ax1.legend(fontsize=7, loc='upper right')

    # A/T/S theo từng đoạn giọng nói, neo vào mốc VAD (đầu/cuối đoạn)
    colors = [COLORS['activation'], COLORS['tension'], COLORS['stability']]
    labels = ['Activation', 'Tension', 'Stability']
    for ax, state, c, lbl in zip(axes[2:], STATE_NAMES, colors, labels):
        for s_start, s_end in speech_segs:
            m = (df['center'] >= s_start) & (df['center'] <= s_end)
            if m.sum() == 0:
                continue
            xs = df.loc[m, 'center'].values
            ys = df.loc[m, f'{state}_smooth'].values
            xs = np.concatenate([[s_start], xs, [s_end]])
            ys = np.concatenate([[ys[0]], ys, [ys[-1]]])
            ax.plot(xs, ys, color=c, lw=2.0)
            ax.fill_between(xs, 0, ys, color=c, alpha=0.15)
        ax.axhspan(MID_LO, MID_HI, color='#ADB5BD', alpha=0.08, zorder=0)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel(lbl + '\n[MODEL]', fontsize=8)
        ax.axhline(0.5, color='#ADB5BD', lw=0.7, ls=':')

    for a in axes:
        a.set_xlim(0, duration)
    axes[-1].set_xlabel('Thời gian (giây)', fontsize=10)
    fig.text(0.5, 0.01,
             'A/T/S vẽ theo từng đoạn giọng nói (đứt ở khoảng lặng) · dải xám ngang = vùng Mid · A/T/S không kết luận to/nhỏ',
             ha='center', fontsize=8, color='#6C757D')
    fig.tight_layout(rect=[0, 0.03, 1, 1])
    return fig


# ── Output 2: Báo cáo kỹ thuật ──────────────────────────────
def make_technical_report(df, raw_info, segments, boundaries) -> str:
    L = ['## Báo cáo kỹ thuật\n']
    L.append(f'- Cửa sổ: {len(df)} (giọng nói {int(df["is_speech"].sum())}) · '
             f'noise floor {raw_info["noise_floor"]:.1f} dB · '
             f'ngưỡng to/nhỏ {raw_info["thresh_loud"]:.1f}/{raw_info["thresh_quiet"]:.1f} dB')
    L.append(f'- Ranh giới Tầng A (embedding âm học): {len(boundaries)} điểm'
             + (f' tại {[f"{b:.1f}s" for b in boundaries]}' if boundaries else ''))
    L.append('')
    L.append('| # | t_start | t_end | dB(RAW) | Tension | Stability | Activation | Mẫu hình | Quỹ đạo | n |')
    L.append('|---|---|---|---|---|---|---|---|---|---|')

    prev = None
    for i, seg in enumerate(segments):
        if not seg['is_speech']:
            # A/T/S không áp dụng cho khoảng lặng → để trống, tránh lẫn với đoạn giọng nói
            L.append(f'| {i+1} | {seg["t_start"]:.1f}s | {seg["t_end"]:.1f}s | 🔇 khoảng lặng | — | — | — | — | — | {seg["n_windows"]} |')
            continue
        pat  = get_vocal_pattern_label(seg, raw_info) or '—'
        traj = classify_trajectory(prev, seg) or '—'
        warn = ' ⚠️' if seg['n_windows'] < 15 else ''
        L.append(f'| {i+1} | {seg["t_start"]:.1f}s | {seg["t_end"]:.1f}s | {seg["voice_level_db_mean"]:.1f} | '
                 f'{seg["tension_mean"]:.2f} | {seg["stability_mean"]:.2f} | {seg["activation_mean"]:.2f} | '
                 f'{pat} | {traj} | {seg["n_windows"]}{warn} |')
        prev = seg
    L.append('\n*🔇 = đoạn không có giọng nói (A/T/S không áp dụng). ⚠️ = n < 15, ước tính kém tin cậy. '
             'Quỹ đạo trống ở đoạn nói đầu tiên hoặc khi không có thay đổi đáng kể. '
             'Độ lớn (RAW) và cách nói (MODEL) là hai sự thật song song.*')
    return '\n'.join(L)


# ── Output 3: Mô tả tự nhiên (VI) ───────────────────────────
def make_natural_description_vi(segments, raw_info) -> str:
    parts = []
    prev = None
    for i, seg in enumerate(segments):
        t = f'[{seg["t_start"]:.1f}s – {seg["t_end"]:.1f}s]'
        parts.append(f'**Đoạn {i+1}** {t}: {describe_segment_vi(seg, raw_info, prev)}')
        if seg['is_speech']:
            prev = seg
    parts.append('\n*Mô tả đặc trưng âm thanh của giọng nói (độ lớn và cách nói), không phải kết luận tâm lý.*')
    return '\n'.join(parts)


# ── Output 4: JSON ──────────────────────────────────────────
def make_json_output(segments, raw_info, df, boundaries) -> str:
    payload = {
        'schema': 'speech_dynamics_studio_v2',
        'raw_branch': {
            'description': 'Waveform GỐC. voice_level_db = RMS_dB − noise_floor. Nguồn duy nhất kết luận to/nhỏ.',
            'noise_floor_db': round(raw_info['noise_floor'], 2),
            'thresh_loud_db': round(raw_info['thresh_loud'], 2),
            'thresh_quiet_db': round(raw_info['thresh_quiet'], 2),
        },
        'model_branch': {
            'description': 'Waveform chuẩn hóa −23 LUFS. A/T/S mô tả cách nói, không kết luận to/nhỏ.',
            'axes': ['activation', 'tension', 'stability'],
            'mid_band': [MID_LO, MID_HI],
        },
        'acoustic_boundaries': [round(b, 3) for b in boundaries],
        'segments': [],
    }
    prev = None
    for seg in segments:
        payload['segments'].append({
            't_start': round(seg['t_start'], 3),
            't_end': round(seg['t_end'], 3),
            'is_speech': seg['is_speech'],
            'raw_branch': {'voice_level_db': round(seg['voice_level_db_mean'], 2)} if seg['is_speech'] else None,
            'model_branch': ({'activation': round(seg['activation_mean'], 3),
                              'tension': round(seg['tension_mean'], 3),
                              'stability': round(seg['stability_mean'], 3)} if seg['is_speech'] else None),
            'vocal_pattern': get_vocal_pattern_label(seg, raw_info),
            'salience': get_dynamics_salience_label(seg),
            'trajectory_vs_prev': classify_trajectory(prev, seg),
            'n_windows': seg['n_windows'],
        })
        if seg['is_speech']:
            prev = seg
    return json.dumps(payload, ensure_ascii=False, indent=2)


# ── Output 5: Voice AI Prompt (EN) ──────────────────────────
def make_voice_ai_prompt(segments, raw_info) -> str:
    L = ['## Voice acting direction', '*Per segment · % relative to previous spoken segment.*\n']
    prev = None
    for i, seg in enumerate(segments):
        if not seg['is_speech']:
            L.append(f'• {describe_segment_en(seg, raw_info)}')
            continue
        line = f'**Seg {i+1}** [{seg["t_start"]:.1f}s–{seg["t_end"]:.1f}s]: {describe_segment_en(seg, raw_info, prev)}'
        if prev:
            dv = seg['voice_level_db_mean'] - prev['voice_level_db_mean']
            if abs(dv) > 2.0:
                pct = min(int(abs(dv) / max(prev['voice_level_db_mean'], 1) * 100), 200)
                line += f' | volume {"up" if dv > 0 else "down"} ~{pct}%'
        L.append(line)
        prev = seg
    return '\n'.join(L)


print('✅ 5 output generators định nghĩa xong')


In [ ]:
# ============================================================
# CELL 11 — Hàm phân tích tổng hợp
# ============================================================

def analyze_audio(audio_input) -> tuple:
    """
    Điểm vào duy nhất từ Gradio.
    audio_input: tuple (sr, np.ndarray) từ gr.Audio.
    Trả về (timeline_fig, report_md, natural_vi, json_str, voice_ai_md).
    """
    if audio_input is None:
        return [None, '⚠️ Chưa có audio.', '', '', '']

    in_sr, wav_in = audio_input
    wav_in = wav_in.astype(np.float32)
    if wav_in.ndim > 1:
        wav_in = wav_in.mean(axis=1)
    # Chuẩn hóa về float [-1, 1] nếu cần
    if wav_in.dtype == np.int16 or np.max(np.abs(wav_in)) > 2.0:
        wav_in = wav_in / (np.max(np.abs(wav_in)) + 1e-8)

    TARGET_SR = AUDIO_CFG['sample_rate']
    if in_sr != TARGET_SR:
        wav_in = librosa.resample(wav_in, orig_sr=in_sr, target_sr=TARGET_SR).astype(np.float32)

    wav_raw   = wav_in.copy()
    wav_model = normalize_lufs(wav_raw, TARGET_SR)

    # Kiểm tra độ dài tối thiểu
    min_sec = WIN_CFG['min_window_sec']
    if len(wav_raw) < int(min_sec * TARGET_SR):
        return [None, f'⚠️ Audio quá ngắn (< {min_sec}s).', '', '', '']

    # VAD
    speech_segs = detect_speech_segments(wav_model, TARGET_SR)

    # Sliding windows trên wav_model
    windows_model = make_windows(wav_model, TARGET_SR)
    windows_raw   = [(s, e, wav_raw[s:e]) for s, e, _ in windows_model]

    if not windows_model:
        return [None, '⚠️ Không trích được cửa sổ.', '', '', '']

    # Nhánh RAW
    raw_info = compute_raw_branch(wav_raw, TARGET_SR, windows_raw)

    # Nhánh MODEL
    df, p1_embs, p2_embs = run_model_branch(windows_model, TARGET_SR, speech_segs)

    # Ranh giới Tầng A
    boundaries = detect_acoustic_boundaries(df, p1_embs)

    # Segments (đã gắn cờ buffer ngắn trong build_segments)
    segments = build_segments(df, raw_info, speech_segs, boundaries)

    # 5 đầu ra (đã bỏ Phase Portrait)
    fig_timeline = make_timeline_figure(df, wav_raw, TARGET_SR, raw_info, speech_segs, boundaries)
    report_md    = make_technical_report(df, raw_info, segments, boundaries)
    natural_vi   = make_natural_description_vi(segments, raw_info)
    json_str     = make_json_output(segments, raw_info, df, boundaries)
    voice_ai_md  = make_voice_ai_prompt(segments, raw_info)

    return [fig_timeline, report_md, natural_vi, json_str, voice_ai_md]


print('✅ analyze_audio() sẵn sàng')


In [ ]:
# ============================================================
# CELL 12 — Giao diện Gradio
# ============================================================

ABOUT_MD = """
### Cách dùng
1. Tải lên hoặc thu một đoạn ghi âm **một người nói** (ít nhạc/ồn nền).
2. Nhấn **Phân tích** → xem kết quả ở các tab bên phải.

### Đọc kết quả
- **Độ lớn (RAW)** — to/nhỏ thật của giọng, đo trên tín hiệu gốc.
- **Cách nói (MODEL)** — sôi nổi / căng / đều nhịp; *không* nói to/nhỏ.

Hai thông tin này tách bạch: một đoạn có thể *nói to* mà vẫn *đều đều*.
"""

with gr.Blocks(title='Speech Dynamics Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎙️ Speech Dynamics Studio')

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            audio_in = gr.Audio(label='File âm thanh (WAV / MP3 / FLAC)',
                                 type='numpy', sources=['upload', 'microphone'])
            run_btn   = gr.Button('🔍 Phân tích', variant='primary', size='lg')
            clear_btn = gr.Button('🗑️ Xóa', size='sm')
            gr.Markdown(ABOUT_MD)

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('📊 Timeline'):
                    out_timeline = gr.Plot(label='Timeline')
                with gr.Tab('💬 Mô tả tự nhiên'):
                    out_natural = gr.Markdown()
                with gr.Tab('📋 Báo cáo kỹ thuật'):
                    out_report = gr.Markdown()
                with gr.Tab('🔧 JSON'):
                    out_json = gr.Code(language='json', label='JSON')
                with gr.Tab('🎭 Voice AI Prompt'):
                    out_voice_ai = gr.Markdown()

    outputs = [out_timeline, out_report, out_natural, out_json, out_voice_ai]
    run_btn.click(fn=analyze_audio, inputs=[audio_in], outputs=outputs)
    clear_btn.click(fn=lambda: [None] * 6, inputs=[], outputs=[audio_in] + outputs)

print('✅ Giao diện Gradio đã build xong')


In [ ]:
# ============================================================
# CELL 13 — Chạy demo
# ============================================================
# share=True tạo link public (hoạt động trong 72h trên Colab)
demo.launch(share=True, debug=False, quiet=True)